In [ ]:
import sqlite3
from pathlib import Path

"""
What is the goal of this file
1. Start the in-memory database for notes metadata which will help us load up all the files
2. Create methods to interact with the in-memory database
"""

# Creating local db and a connection to it
conn = sqlite3.connect(':memory:')
c = conn.cursor()


# Adding a table the local DB
c.execute("""CREATE TABLE notes (
                note_id TEXT PRIMARY KEY,
                note_title TEXT,
                note_version TEXT,
                note_dir TEXT)""")


#Local DB Methods
def add_metadata(note: Note) -> bool:
    nt = note.__dict__
    nt['note_dir'] = str(note.note_dir)
    try:
        c.execute("INSERT INTO notes VALUES (:note_id, :note_title, :note_version, :note_dir)", nt)
        conn.commit()
        return True
    except sqlite3.IntegrityError:
        return False

def get_metadata(nt_id: str) -> Note:
    c.execute("SELECT * FROM notes WHERE note_id = ?", (nt_id,))
    nt = c.fetchone()
    return Note(nt[0], nt[1], nt[2], Path(nt[3]))


# Defining Note class to handle the metadata properly
class Note:
    def __init__(self, note_id: str, note_title: str, note_version: str, note_dir: Path):
        self.note_id = note_id
        self.note_title = note_title
        self.note_version = note_version
        self.note_dir = note_dir

    # Method to validate a note (will add later)
    def validate_note(self):
        pass

    def __str__(self):
        tpl = (self.note_id, self.note_title, self.note_version, str(self.note_dir))
        return str(tpl)


In [ ]:
ntPath = Path("C:/Users/ADMIN/Development/PyTauri/project sushi sandbox-vault/11f5dd80-a1b1-4381-8fc0-7700cf65b627.jnote")
nt1 = Note('11f5dd80-a1b1-4381-8fc0-7700cf65b627','My First Deep Learning Note','1.0',ntPath)

In [ ]:
c.execute("SELECT * FROM notes")
c.fetchall()

In [ ]:
nt1.__dict__

In [ ]:
add_metadata(nt1)
c.execute("SELECT * FROM notes")
c.fetchall()

In [ ]:
print(get_metadata("11f5dd80-a1b1-4381-8fc0-7700cf65b627"))

In [ ]:
c.execute("SELECT * FROM notes")
c.fetchall()

In [1]:
import sqlite3
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import Optional, List

@dataclass
class Note:
    note_id: str
    note_title: str
    note_version: str
    note_dir: Path

    def __post_init__(self):
        # Ensure note_dir is always a Path object
        if isinstance(self.note_dir, str):
            self.note_dir = Path(self.note_dir)

class NoteIndex:
    def __init__(self, db_path: str = ":memory:"):
        # check_same_thread=False is often needed for GUI frameworks like Tauri
        self.conn = sqlite3.connect(db_path, check_same_thread=False)
        self.cursor = self.conn.cursor()
        self._setup_db()

    def _setup_db(self):
        self.cursor.execute("""
            CREATE TABLE IF NOT EXISTS notes (
                note_id TEXT PRIMARY KEY,
                note_title TEXT,
                note_version TEXT,
                note_dir TEXT
            )
        """)
        self.conn.commit()

    def add_metadata(self, note: Note) -> bool:
        try:
            # We convert the dataclass to a dict and handle the Path object
            data = asdict(note)
            data['note_dir'] = str(data['note_dir'])

            self.cursor.execute("""
                INSERT INTO notes (note_id, note_title, note_version, note_dir)
                VALUES (:note_id, :note_title, :note_version, :note_dir)
            """, data)
            self.conn.commit()
            return True
        except sqlite3.IntegrityError:
            return False

    def get_metadata(self, nt_id: str) -> Optional[Note]:
        self.cursor.execute("SELECT * FROM notes WHERE note_id = ?", (nt_id,))
        row = self.cursor.fetchone()

        if row:
            return Note(
                note_id=row[0],
                note_title=row[1],
                note_version=row[2],
                note_dir=Path(row[3])
            )
        return None

    def get_all_notes(self) -> List[Note]:
        self.cursor.execute("SELECT * FROM notes")
        return [Note(r[0], r[1], r[2], Path(r[3])) for r in self.cursor.fetchall()]

In [5]:
ntPath = Path("C:/Users/ADMIN/Development/PyTauri/project sushi sandbox-vault/11f5dd80-a1b1-4381-8fc0-7700cf65b627.jnote")
nt1 = Note('11f5dd80-a1b1-4381-8fc0-7700cf65b627','My First Deep Learning Note','1.0',ntPath)

In [8]:
localdb = NoteIndex()

In [9]:
localdb.add_metadata(nt1)

True

In [17]:
localdb.get_metadata("11f5dd80-a1b1-4381-8fc0-7700cf65b627")

Note(note_id='11f5dd80-a1b1-4381-8fc0-7700cf65b627', note_title='My First Deep Learning Note', note_version='1.0', note_dir=WindowsPath('C:/Users/ADMIN/Development/PyTauri/project sushi sandbox-vault/11f5dd80-a1b1-4381-8fc0-7700cf65b627.jnote'))

In [18]:
localdb.get_all_notes()

[Note(note_id='11f5dd80-a1b1-4381-8fc0-7700cf65b627', note_title='My First Deep Learning Note', note_version='1.0', note_dir=WindowsPath('C:/Users/ADMIN/Development/PyTauri/project sushi sandbox-vault/11f5dd80-a1b1-4381-8fc0-7700cf65b627.jnote'))]

In [19]:
ntPath = Path("C:/Users/ADMIN/Development/PyTauri/project sushi sandbox-vault/b60af6f0-ac35-445f-bc2c-62df8eaa3ba6.jnote")
nt2 = Note('b60af6f0-ac35-445f-bc2c-62df8eaa3ba6','My Second Deep Learning Note','1.0',ntPath)
localdb.add_metadata(nt2)
localdb.get_all_notes()

[Note(note_id='11f5dd80-a1b1-4381-8fc0-7700cf65b627', note_title='My First Deep Learning Note', note_version='1.0', note_dir=WindowsPath('C:/Users/ADMIN/Development/PyTauri/project sushi sandbox-vault/11f5dd80-a1b1-4381-8fc0-7700cf65b627.jnote')),
 Note(note_id='b60af6f0-ac35-445f-bc2c-62df8eaa3ba6', note_title='My Second Deep Learning Note', note_version='1.0', note_dir=WindowsPath('C:/Users/ADMIN/Development/PyTauri/project sushi sandbox-vault/b60af6f0-ac35-445f-bc2c-62df8eaa3ba6.jnote'))]

In [1]:
import ijson

file_path = "b60af6f0-ac35-445f-bc2c-62df8eaa3ba6.jnote"

def extract_note_metadata(file_path):
    """
    Efficiently extracts the 'metadata' object from the note JSON
    without parsing the rest of the file.
    """
    metadata = {}

    try:
        with open(file_path, 'rb') as f:
            # ijson.items allows us to target a specific prefix.
            # 'metadata' matches the key in your JSON.
            objects = ijson.items(f, 'metadata')

            # Since 'metadata' is a single object, we take the first item
            for obj in objects:
                metadata = obj
                break # Exit early once the metadata object is parsed

    except Exception as e:
        print(f"Error reading {file_path}: {e}")

    return metadata



In [8]:
raw_data = extract_note_metadata(file_path)
raw_data

{'title': 'My First Deep Learning Note',
 'created_at': '2025-12-26T06:21:01.585221',
 'last_modified': '2025-12-26T06:21:01.585221',
 'version': Decimal('1.0'),
 'note_id': 'b60af6f0-ac35-445f-bc2c-62df8eaa3ba6',
 'status': 0,
 'tags': []}

In [13]:
str(raw_data.get("version"))

'1.0'